In [72]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/across_cohort_descriptive_func.R')
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')

path.in <- '/home/jupyter/workspaces/Data/'
path.out <- '/home/jupyter/workspaces/result/final/'

In [52]:
diabetes.pid <- read.delim(
    paste0(path.in, 'diabetes_final_pid.txt'),
    as.is = T
);
insomnia.pid <- read.delim(
    paste0(path.in, 'insomnia_final_pid.txt'),
    as.is = T
);
pca.pid <- read.delim(
    paste0(path.in, 'prostate_final_pid.txt'),
    as.is = T
);
healthy.pid <- read.delim(
    paste0(path.in, 'healthy_final_pid.txt'),
    as.is = T
);
length(diabetes.pid$x)

[1] 1547

In [25]:
# fitbit activity
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(diabetes.pid$x, collapse = ", ")
diabetes_activity_sql <- paste("
       SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary  
    WHERE
        activity_summary.PERSON_ID IN (", diabetes.pid.sql, ")"
)

diabetes_activity_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_activity",
  "diabetes_activity_*.csv")
message(str_glue('The data will be written to {diabetes_activity_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_activity_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_activity/diabetes_activity_*.csv. Use this path when reading the data into your notebooks in the future.

Auto-refreshing stale OAuth token.



In [26]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_activity_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_activity/diabetes_activity_*.csv'

diabetes_activity <- read_bq_export_from_workspace_bucket(diabetes_activity_path)

dim(diabetes_activity)

head(diabetes_activity, 5) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_activity/diabetes_activity_000000000000.csv.



[1] 529621     13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3273928,2023-07-19,303,1586,1878,NA,0,NA,72,169,178,2459,0
3273928,2023-09-10,124,1586,1674,NA,0,NA,39,55,0,749,0
3273928,2023-07-21,260,1586,1778,NA,0,NA,76,115,322,1853,0
3273928,2023-07-27,309,1586,1839,NA,0,NA,87,146,729,2330,0
3273928,2023-09-13,388,1586,1892,NA,0,NA,111,176,344,2759,0


In [27]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_meta/diabetes_meta_000000000000.csv'

diabetes_meta <- read_bq_export_from_workspace_bucket(diabetes_meta_path)

dim(diabetes_meta)

head(diabetes_meta, 3) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_meta/diabetes_meta_000000000000.csv.



[1] 1547   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1344748,1966-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
1617237,1964-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
2977576,1968-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population


In [28]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Diabetes/', df = diabetes_activity)

── Wear compliance ──
     n_days       followup_days   
 Min.   :   1.0   Min.   :   1.0  
 1st Qu.:  93.0   1st Qu.:  95.0  
 Median : 109.0   Median : 111.0  
 Mean   : 342.4   Mean   : 347.6  
 3rd Qu.: 351.0   3rd Qu.: 352.0  
 Max.   :3898.0   Max.   :3967.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003770   Min.   :   1.0   Min.   :2012-11-21   Min.   :2015-10-18  
 1st Qu.:1917516   1st Qu.:  93.0   1st Qu.:2022-06-24   1st Qu.:2023-10-01  
 Median :3240839   Median : 109.0   Median :2023-06-12   Median :2023-10-01  
 Mean   :4245703   Mean   : 342.4   Mean   :2022-08-20   Mean   :2023-08-02  
 3rd Qu.:6371856   3rd Qu.: 351.0   3rd Qu.:2023-06-25   3rd Qu.:2023-10-01  
 Max.   :9987610   Max.   :3898.0   Max.   :2023-09-24   Max.   :2023-10-01  
 followup_days     missing_days      pct_missing        compliance_tier
 Min.   :   1.0   Min.   :  0.000   Min.   : 0.000   High (>90%):1487  
 1st Qu.:  95.0   1st Qu.:  0.000   1st Qu.: 0.000   80-90%     :  53  
 Median : 111.0   Median :  0.000   Median : 0.000   70-80%     :   7  
 Mean   : 347.6   Mean   :  5.197   Mean   : 1.675   60-70%     :   0  
 3rd Qu.: 352.0   3rd Qu.:  2.000   3rd Qu.: 1.822   50-60%     :   0  
 Max.   :3967.0   Max.

NULL

In [29]:
diabetes_patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Diabetes/',
    cohort = 'Diabetes', df = diabetes_activity,
    meta.data = diabetes_meta[,c('person_id', 'sex_at_birth')])

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out            1547 2313.   575.  2.25e3 1.90e3 2649.  1.52e+3 3.37e3
2 fairly_active_minutes   1547   11.8   12.5 8.27e0 2.73e0   16.5 2.75e-1 3.57e1
3 lightly_active_minutes  1547  171.    75.5 1.69e2 1.19e2  221.  4.95e+1 2.99e2
4 sedentary_minutes       1547  840.   167.  8.17e2 7.32e2  904.  6.27e+2 1.20e3
5 steps                   1547 5043.  3314.  4.49e3 2.54e3 6803.  8.96e+2 1.13e4
6 very_active_minutes     1547   11.5   16.1 5.32e0 1.39e0   14.7 1.29e-1 4.61e1


In [30]:
# fitbit activity
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(insomnia.pid$x, collapse = ", ")
insomnia_activity_sql <- paste("
       SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary  
    WHERE
        activity_summary.PERSON_ID IN (", insomnia.pid.sql, ")"
)

insomnia_activity_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_activity",
  "insomnia_activity_*.csv")
message(str_glue('The data will be written to {insomnia_activity_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_activity_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_activity/insomnia_activity_*.csv. Use this path when reading the data into your notebooks in the future.



In [32]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_activity_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_activity/insomnia_activity_*.csv'

insomnia_activity <- read_bq_export_from_workspace_bucket(insomnia_activity_path)

dim(insomnia_activity)

head(insomnia_activity, 2) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_activity/insomnia_activity_000000000000.csv.



[1] 598773     13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2473759,2022-07-04,0,NA,1406,NA,0,NA,0,NA,1440,0,0
2473759,2022-07-06,0,NA,1406,NA,0,NA,0,NA,1440,0,0


In [31]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_meta/insomnia_meta_*.csv'

insomnia_meta <- read_bq_export_from_workspace_bucket(insomnia_meta_path)

dim(insomnia_meta)

head(insomnia_meta, 2) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_meta/insomnia_meta_000000000000.csv.



[1] 1625   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1107258,1942-06-15 00:00:00 UTC,2.1e+09,None Indicated,38003563,Hispanic or Latino,45880669,Male,1586147,What Race Ethnicity: Hispanic
1598808,1959-06-15 00:00:00 UTC,2.1e+09,None Indicated,38003563,Hispanic or Latino,45880669,Male,1586147,What Race Ethnicity: Hispanic


In [33]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Insomnia/', df = insomnia_activity)

── Wear compliance ──
     n_days       followup_days   
 Min.   :   1.0   Min.   :   1.0  
 1st Qu.:  90.0   1st Qu.:  91.0  
 Median : 108.0   Median : 109.0  
 Mean   : 368.5   Mean   : 374.5  
 3rd Qu.: 403.0   3rd Qu.: 412.0  
 Max.   :5386.0   Max.   :5386.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1002177   Min.   :   1.0   Min.   :2009-01-02   Min.   :2017-11-21  
 1st Qu.:1873007   1st Qu.:  90.0   1st Qu.:2022-06-08   1st Qu.:2023-10-01  
 Median :3136309   Median : 108.0   Median :2023-06-12   Median :2023-10-01  
 Mean   :4161273   Mean   : 368.5   Mean   :2022-07-15   Mean   :2023-07-23  
 3rd Qu.:6344328   3rd Qu.: 403.0   3rd Qu.:2023-06-26   3rd Qu.:2023-10-01  
 Max.   :9987114   Max.   :5386.0   Max.   :2023-09-29   Max.   :2023-10-01  
 followup_days     missing_days      pct_missing        compliance_tier
 Min.   :   1.0   Min.   :  0.000   Min.   : 0.000   High (>90%):1571  
 1st Qu.:  91.0   1st Qu.:  0.000   1st Qu.: 0.000   80-90%     :  47  
 Median : 109.0   Median :  0.000   Median : 0.000   70-80%     :   7  
 Mean   : 374.5   Mean   :  6.012   Mean   : 1.653   60-70%     :   0  
 3rd Qu.: 412.0   3rd Qu.:  3.000   3rd Qu.: 1.826   50-60%     :   0  
 Max.   :5386.0   Max.

NULL

In [34]:
insomnia_patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Insomnia/',
    cohort = 'Insomnia', df = insomnia_activity,
    meta.data = insomnia_meta[,c('person_id', 'sex_at_birth')])

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out            1625 2187.   568.  2.11e3 1.76e3 2522.  1.42e+3 3.20e3
2 fairly_active_minutes   1625   13.1   13.4 9.31e0 3.93e0   18.1 3.22e-1 3.76e1
3 lightly_active_minutes  1625  181.    78.2 1.83e2 1.29e2  230.  4.97e+1 3.07e2
4 sedentary_minutes       1625  818.   179.  7.86e2 7.08e2  889.  5.92e+2 1.22e3
5 steps                   1625 5597.  3453.  5.15e3 2.99e3 7490.  9.42e+2 1.21e4
6 very_active_minutes     1625   13.0   16.6 6.92e0 2   e0   17.8 1.32e-1 4.64e1


In [35]:
# fitbit activity
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(pca.pid$x, collapse = ", ")
pca_activity_sql <- paste("
       SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary  
    WHERE
        activity_summary.PERSON_ID IN (", pca.pid.sql, ")"
)

pca_activity_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_activity",
  "pca_activity_*.csv")
message(str_glue('The data will be written to {pca_activity_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_activity_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_activity/pca_activity_*.csv. Use this path when reading the data into your notebooks in the future.



In [36]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_activity_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_activity/pca_activity_*.csv'

pca_activity <- read_bq_export_from_workspace_bucket(pca_activity_path)

dim(pca_activity)

head(pca_activity, 2) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_activity/pca_activity_000000000000.csv.



[1] 105817     13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1956459,2023-07-10,494,1478,1870,NA,0,NA,159,226,891,3294,0
1956459,2023-09-01,414,1478,1811,NA,0,NA,130,186,846,3111,0


In [37]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_meta/pca_meta_*.csv'

pca_meta <- read_bq_export_from_workspace_bucket(pca_meta_path)

dim(pca_meta)

head(pca_meta, 2)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_meta/pca_meta_000000000000.csv.



[1] 281  10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
2967502,1953-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,45882607,None of these
2462406,1948-06-15 00:00:00 UTC,8527,White,38003563,Hispanic or Latino,45880669,Male,2000000008,More than one population


In [38]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Prostate/', df = pca_activity)

── Wear compliance ──
     n_days       followup_days   
 Min.   :   4.0   Min.   :   4.0  
 1st Qu.:  91.0   1st Qu.:  92.0  
 Median : 104.0   Median : 104.0  
 Mean   : 376.6   Mean   : 383.3  
 3rd Qu.: 243.0   3rd Qu.: 243.0  
 Max.   :3491.0   Max.   :3524.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003895   Min.   :   4.0   Min.   :2014-01-17   Min.   :2020-06-21  
 1st Qu.:1898527   1st Qu.:  91.0   1st Qu.:2023-01-28   1st Qu.:2023-10-01  
 Median :3355707   Median : 104.0   Median :2023-06-17   Median :2023-10-01  
 Mean   :4280805   Mean   : 376.6   Mean   :2022-08-23   Mean   :2023-09-09  
 3rd Qu.:6641809   3rd Qu.: 243.0   3rd Qu.:2023-06-29   3rd Qu.:2023-10-01  
 Max.   :9942771   Max.   :3491.0   Max.   :2023-09-20   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :   4.0   Min.   :  0.00   Min.   : 0.000   High (>90%):270   
 1st Qu.:  92.0   1st Qu.:  0.00   1st Qu.: 0.000   80-90%     : 10   
 Median : 104.0   Median :  0.00   Median : 0.000   70-80%     :  1   
 Mean   : 383.3   Mean   :  6.74   Mean   : 1.452   60-70%     :  0   
 3rd Qu.: 243.0   3rd Qu.:  2.00   3rd Qu.: 1.159   50-60%     :  0   
 Max.   :3524.0   Max.   :17

NULL

In [39]:
pca_patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Prostate/',
    cohort = 'Prostate', df = pca_activity,
    meta.data = pca_meta[,c('person_id', 'sex_at_birth')])

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out             281 2503.   421.  2478.  2.21e3 2769.  1.89e+3 3.25e3
2 fairly_active_minutes    281   18.8   16.2   14.4 6.55e0   26.9 1.20e+0 5.05e1
3 lightly_active_minutes   281  183.    65.1  188.  1.36e2  227.  7.72e+1 2.81e2
4 sedentary_minutes        281  787.   139.   775.  6.94e2  857.  5.98e+2 1.05e3
5 steps                    281 6861.  3921.  6374.  4.36e3 9094.  1.39e+3 1.26e4
6 very_active_minutes      281   22.2   22.1   16.3 5.34e0   29.4 4.85e-1 6.63e1


In [40]:
# fitbit activity
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(healthy.pid$x, collapse = ", ")
healthy_activity_sql <- paste("
       SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary  
    WHERE
        activity_summary.PERSON_ID IN (", healthy.pid.sql, ")"
)

healthy_activity_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_activity",
  "healthy_activity_*.csv")
message(str_glue('The data will be written to {healthy_activity_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_activity_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_activity/healthy_activity_*.csv. Use this path when reading the data into your notebooks in the future.



In [41]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

healthy_activity_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_activity/healthy_activity_*.csv'

healthy_activity <- read_bq_export_from_workspace_bucket(healthy_activity_path)

dim(healthy_activity)

head(healthy_activity, 2)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_activity/healthy_activity_000000000000.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_activity/healthy_activity_000000000001.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_activity/healthy_activity_000000000002.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_activity/healthy_activity_000000000003.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_activity/healthy_activity_000000000004.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_activity/healthy_activity_000000000005.csv.



[1] 3376604      13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2078603,2023-09-12,573,1813,2402,NA,8,NA,123,288,840,2397,0
2078603,2023-08-18,680,1813,2447,NA,0,NA,169,312,892,2761,0


In [42]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_meta/healthy_meta_*.csv'

healthy_meta <- read_bq_export_from_workspace_bucket(healthy_meta_path)

dim(healthy_meta)

head(healthy_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_meta/healthy_meta_000000000000.csv.



[1] 6960   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
4337953,1965-06-15 00:00:00 UTC,8557,Native Hawaiian or Other Pacific Islander,38003564,Not Hispanic or Latino,45878463,Female,8557,Native Hawaiian or Other Pacific Islander
1188079,1981-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
1346705,1998-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
3043509,1997-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
2523178,1992-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these


In [43]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Healthy/', df = healthy_activity)

── Wear compliance ──
     n_days       followup_days   
 Min.   :   1.0   Min.   :   1.0  
 1st Qu.:  97.0   1st Qu.:  98.0  
 Median : 128.0   Median : 130.0  
 Mean   : 485.1   Mean   : 491.3  
 3rd Qu.: 706.2   3rd Qu.: 709.0  
 Max.   :4154.0   Max.   :4198.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1000464   Min.   :   1.0   Min.   :2012-04-04   Min.   :2016-02-05  
 1st Qu.:2285677   1st Qu.:  97.0   1st Qu.:2021-07-26   1st Qu.:2023-10-01  
 Median :4257414   Median : 128.0   Median :2023-04-25   Median :2023-10-01  
 Mean   :4753837   Mean   : 485.1   Mean   :2022-03-24   Mean   :2023-07-27  
 3rd Qu.:7090724   3rd Qu.: 706.2   3rd Qu.:2023-06-21   3rd Qu.:2023-10-01  
 Max.   :9997301   Max.   :4154.0   Max.   :2023-09-30   Max.   :2023-10-01  
 followup_days     missing_days      pct_missing        compliance_tier
 Min.   :   1.0   Min.   :  0.000   Min.   : 0.000   High (>90%):6741  
 1st Qu.:  98.0   1st Qu.:  0.000   1st Qu.: 0.000   80-90%     : 188  
 Median : 130.0   Median :  0.000   Median : 0.000   70-80%     :  31  
 Mean   : 491.3   Mean   :  6.152   Mean   : 1.429   60-70%     :   0  
 3rd Qu.: 709.0   3rd Qu.:  2.000   3rd Qu.: 1.418   50-60%     :   0  
 Max.   :4198.0   Max.

NULL

In [44]:
healthy_patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Healthy/',
    cohort = 'Healthy', df = healthy_activity,
    meta.data = healthy_meta[,c('person_id', 'sex_at_birth')])

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out            6960 2262.   597.  2179.  1.83e3 2629.  1.43e+3 3.34e3
2 fairly_active_minutes   6960   16.1   15.5   12.1 5.62e0   21.9 6.13e-1 4.44e1
3 lightly_active_minutes  6960  193.    83.3  195.  1.42e2  247.  3.72e+1 3.26e2
4 sedentary_minutes       6960  809.   202.   767.  6.83e2  868.  5.75e+2 1.30e3
5 steps                   6960 6707.  4078.  6261.  3.80e3 9036.  8.10e+2 1.37e4
6 very_active_minutes     6960   18.2   21.5   11.1 3.62e0   25.3 2.66e-1 5.85e1


In [53]:
#### sleep ########
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(diabetes.pid$x, collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
diabetes_sleep_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (", diabetes.pid.sql, ")"
)
# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
diabetes_sleep_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_sleep",
  "diabetes_sleep.csv")
message(str_glue('The data will be written to {diabetes_sleep_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_sleep_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_sleep_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_sleep/diabetes_sleep.csv. Use this path when reading the data into your notebooks in the future.



In [54]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_sleep_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_sleep/diabetes_sleep.csv'

diabetes_sleep <- read_bq_export_from_workspace_bucket(diabetes_sleep_path)

dim(diabetes_sleep)

head(diabetes_sleep, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_sleep/diabetes_sleep.csv.



[1] 539434     12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
8215159,2022-04-11,false,182,160,0,22,NA,6,112,42,22
8170087,2022-01-30,false,183,160,0,23,NA,45,87,28,23
8203187,2023-07-09,true,182,161,0,21,NA,33,108,20,21
8230713,2021-06-24,false,182,161,3,21,NA,18,137,6,21
8263432,2023-07-22,true,184,161,21,23,NA,0,133,28,23


In [55]:
colnames(diabetes_sleep)[2] <- 'date'
diabetes_sleep_main <- diabetes_sleep |>
  filter(is_main_sleep == "true")

diabetes_sleep_nap <- diabetes_sleep |>
  filter(is_main_sleep == "false")

diabetes_sleep_sum <- sleep_sum_main_nap_func(df = diabetes_sleep)

In [57]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Diabetes/sleep/', df = diabetes_sleep_sum)

── Wear compliance ──
     n_days       followup_days   
 Min.   :   1.0   Min.   :   1.0  
 1st Qu.:  68.0   1st Qu.:  82.0  
 Median :  91.0   Median : 104.0  
 Mean   : 289.6   Mean   : 311.1  
 3rd Qu.: 225.5   3rd Qu.: 250.5  
 Max.   :3881.0   Max.   :3966.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003770   Min.   :   1.0   Min.   :2012-11-22   Min.   :2015-10-18  
 1st Qu.:1917516   1st Qu.:  68.0   1st Qu.:2022-11-01   1st Qu.:2023-09-25  
 Median :3240839   Median :  91.0   Median :2023-06-15   Median :2023-10-01  
 Mean   :4245703   Mean   : 289.6   Mean   :2022-09-14   Mean   :2023-07-21  
 3rd Qu.:6371856   3rd Qu.: 225.5   3rd Qu.:2023-07-03   3rd Qu.:2023-10-01  
 Max.   :9987610   Max.   :3881.0   Max.   :2023-09-30   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :   1.0   Min.   :  0.00   Min.   : 0.000   High (>90%):947   
 1st Qu.:  82.0   1st Qu.:  3.00   1st Qu.: 3.030   80-90%     :381   
 Median : 104.0   Median : 11.00   Median : 7.143   70-80%     :219   
 Mean   : 311.1   Mean   : 21.54   Mean   : 9.507   60-70%     :  0   
 3rd Qu.: 250.5   3rd Qu.: 23.00   3rd Qu.:14.523   50-60%     :  0   
 Max.   :3966.0   Max.   :37

NULL

In [58]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Diabetes/sleep/',
    cohort = 'Diabetes',
    df = diabetes_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Diabetes/sleep/',
    cohort = 'Diabetes',
    df = diabetes_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep  1547 368.   72.6  373.  328.  414.  237.  475. 
2 minute_awake   1547  52.0  14.9   51.6  43.1  60.7  27.7  74.7
3 minute_deep    1534  53.0  16.2   52.4  40.9  63.9  28.4  79.9
4 minute_in_bed  1547 421.   81.6  428.  375.  473.  273.  540. 
5 minute_light   1534 257.   47.4  254.  226.  286.  184.  338. 
6 minute_rem     1534  71.2  22.5   71.0  56.7  85.9  35.4 107. 
7 minute_wake    1534  55.2  12.6   54.4  46.8  63.1  36.7  76.4
# A tibble: 7 × 9
  metric            N  Mean    SD Median     Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl>  <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep  1492 104.  29.9  102.    87.4  117.   65.6  148. 
2 minute_awake   1492  11.9  8.84   9.36   6.45  13.6   3.49  32.0
3 minute_deep    1093  23.8 11.5   23.6   16.2   30.6   5.87  42.4
4 minute_in_bed  1492 117.  31.3  113.    98

In [59]:
head(diabetes_sleep_sum)
patient_stats_sum <-sleep_prep_stats(df = diabetes_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Diabetes/sleep/', cohort = 'Diabetes', main = "sum")

person_id,date,has_main_sleep,has_nap,n_records,minute_in_bed,minute_asleep,minute_awake,minute_restless,minute_deep,⋯,minute_wake,minute_after_wakeup,main_minute_in_bed,main_minute_asleep,main_minute_awake,main_minute_deep,main_minute_light,main_minute_rem,main_minute_wake,sleep_efficiency
<dbl>,<date>,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1003770,2023-08-31,TRUE,FALSE,1,294,256,38,0,52,⋯,38,0,294,256,38,52,152,52,38,87.07483
1003770,2023-09-01,TRUE,FALSE,1,151,148,3,3,0,⋯,0,0,151,148,3,0,0,0,0,98.01325
1003770,2023-09-02,TRUE,FALSE,1,344,319,25,0,34,⋯,25,5,344,319,25,34,211,74,25,92.73256
1003770,2023-09-03,TRUE,TRUE,2,377,349,28,0,57,⋯,28,0,377,349,28,57,122,87,28,92.57294
1003770,2023-09-04,TRUE,FALSE,1,213,191,22,0,42,⋯,22,0,213,191,22,42,86,63,22,89.67136
1003770,2023-09-05,TRUE,FALSE,1,150,149,1,1,0,⋯,0,0,150,149,1,0,0,0,0,99.33333


# A tibble: 8 × 9
  metric               N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>            <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep     1547 390.  71.6   393.  350.  430.  269.  493. 
2 minute_awake      1547  54.2 15.5    53.6  44.9  63.5  30.1  78.7
3 minute_deep       1547  47.0 18.6    47.1  35.0  59.5  15.1  76.3
4 minute_in_bed     1547 445.  79.8   448.  402.  489.  315.  565. 
5 minute_light      1547 226.  68.6   234.  192.  270.   97.0 323. 
6 minute_rem        1547  63.3 25.6    64.8  47.2  80.5  18.5 102. 
7 minute_wake       1547  48.8 16.4    49.8  40.1  59.2  19.6  72.7
8 sleep_efficiency  1547  87.7  3.34   88.1  86.5  89.5  83.2  91.7


In [66]:
#### sleep ########
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(insomnia.pid$x, collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
insomnia_sleep_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (", insomnia.pid.sql, ")"
)
# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
insomnia_sleep_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_sleep",
  "insomnia_sleep.csv")
message(str_glue('The data will be written to {insomnia_sleep_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_sleep_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_sleep_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_sleep/insomnia_sleep.csv. Use this path when reading the data into your notebooks in the future.



In [61]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_sleep_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_sleep/insomnia_sleep.csv'

insomnia_sleep <- read_bq_export_from_workspace_bucket(insomnia_sleep_path)

dim(insomnia_sleep)

head(insomnia_sleep, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_sleep/insomnia_sleep.csv.



[1] 589822     12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2764133,2022-02-24,true,182,164,0,18,NA,16,93,55,18
2791777,2021-09-21,true,182,158,0,24,NA,7,131,20,24
2791777,2021-08-28,true,184,169,0,15,NA,5,141,23,15
2756524,2023-01-28,true,185,169,0,16,NA,26,118,25,16
2764133,2022-02-02,true,186,171,0,15,NA,11,118,42,15


In [68]:
colnames(insomnia_sleep)[2] <- 'date'
insomnia_sleep_main <- insomnia_sleep |>
  filter(is_main_sleep == "true")

insomnia_sleep_nap <- insomnia_sleep |>
  filter(is_main_sleep == "false")

insomnia_sleep_sum <- sleep_sum_main_nap_func(df = insomnia_sleep)

In [69]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Insomnia/sleep/', df = insomnia_sleep_sum)

── Wear compliance ──
     n_days     followup_days   
 Min.   :   1   Min.   :   1.0  
 1st Qu.:  64   1st Qu.:  69.0  
 Median :  90   Median : 102.0  
 Mean   : 308   Mean   : 329.3  
 3rd Qu.: 238   3rd Qu.: 260.0  
 Max.   :4166   Max.   :4364.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days       first_date           last_date         
 Min.   :1002177   Min.   :   1   Min.   :2011-09-30   Min.   :2017-11-20  
 1st Qu.:1873007   1st Qu.:  64   1st Qu.:2022-07-25   1st Qu.:2023-09-22  
 Median :3136309   Median :  90   Median :2023-06-15   Median :2023-10-01  
 Mean   :4161273   Mean   : 308   Mean   :2022-08-16   Mean   :2023-07-11  
 3rd Qu.:6344328   3rd Qu.: 238   3rd Qu.:2023-07-03   3rd Qu.:2023-10-01  
 Max.   :9987114   Max.   :4166   Max.   :2023-09-30   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :   1.0   Min.   :  0.00   Min.   : 0.000   High (>90%):1034  
 1st Qu.:  69.0   1st Qu.:  3.00   1st Qu.: 2.497   80-90%     : 389  
 Median : 102.0   Median : 10.00   Median : 6.593   70-80%     : 202  
 Mean   : 329.3   Mean   : 21.33   Mean   : 8.978   60-70%     :   0  
 3rd Qu.: 260.0   3rd Qu.: 22.00   3rd Qu.:13.992   50-60%     :   0  
 Max.   :4364.0   Max.   :316.00   Max.   

NULL

In [70]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Insomnia/sleep/',
    cohort = 'Insomnia',
    df = insomnia_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Insomnia/sleep/',
    cohort = 'Insomnia',
    df = insomnia_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep  1625 380.   75.7  388.  337.  427.  240.  489. 
2 minute_awake   1625  53.8  15.7   53.7  44.8  62    30.6  78.2
3 minute_deep    1612  56.0  17.0   56.1  43.5  67.4  28.9  83.8
4 minute_in_bed  1625 434.   84.7  445.  386.  486.  281.  555. 
5 minute_light   1612 261.   48.1  260.  229.  292.  187.  342. 
6 minute_rem     1612  74.8  23.2   74.9  59.2  90.0  36.5 112. 
7 minute_wake    1612  56.2  12.6   55.6  47.9  63.4  37.5  78.1
# A tibble: 7 × 9
  metric            N  Mean    SD Median     Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl>  <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep  1528 103.  28.9   101.   86.0  117.   66.7  146. 
2 minute_awake   1528  12.8  9.86    9.6   6.62  15.0   3.37  35.1
3 minute_deep    1083  24.3 12.1    24    16.6   31.2   5     45.6
4 minute_in_bed  1528 117.  30.8   112    98

In [71]:
head(insomnia_sleep_sum)
patient_stats_sum <-sleep_prep_stats(df = insomnia_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Insomnia/sleep/', cohort = 'Insomnia', main = "sum")

person_id,date,has_main_sleep,has_nap,n_records,minute_in_bed,minute_asleep,minute_awake,minute_restless,minute_deep,⋯,minute_wake,minute_after_wakeup,main_minute_in_bed,main_minute_asleep,main_minute_awake,main_minute_deep,main_minute_light,main_minute_rem,main_minute_wake,sleep_efficiency
<dbl>,<date>,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1002177,2022-02-14,TRUE,FALSE,1,150,140,10,10,0,⋯,0,0,150,140,10,0,0,0,0,93.33333
1003126,2014-02-16,TRUE,FALSE,1,541,530,11,9,0,⋯,0,0,541,530,11,0,0,0,0,97.96673
1003126,2014-02-17,TRUE,FALSE,1,489,486,3,3,0,⋯,0,0,489,486,3,0,0,0,0,99.38650
1003126,2014-02-18,TRUE,FALSE,1,461,444,17,15,0,⋯,0,0,461,444,17,0,0,0,0,96.31236
1003126,2014-02-19,TRUE,FALSE,1,508,494,14,12,0,⋯,0,0,508,494,14,0,0,0,0,97.24409
1003126,2014-02-20,TRUE,FALSE,1,505,485,20,16,0,⋯,0,0,505,485,20,0,0,0,0,96.03960


# A tibble: 8 × 9
  metric               N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>            <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep     1625 398.  74.4   404.  359.  441.  270.  508. 
2 minute_awake      1625  56.0 16.6    55.7  46.2  65.0  31.5  81.2
3 minute_deep       1625  50.4 19.4    51.2  37.5  63.9  17.5  80.1
4 minute_in_bed     1625 455.  83.9   462.  408.  503.  318.  576. 
5 minute_light      1625 234.  68.4   244.  201.  277.  110.  328. 
6 minute_rem        1625  67.3 26.2    68.2  50.3  85.3  20.8 107. 
7 minute_wake       1625  50.5 16.3    51.6  41.0  60.2  22.7  76.5
8 sleep_efficiency  1625  87.6  3.53   88.0  86.4  89.4  82.8  91.7
